# 03 — Full-text extraction

**Goal:** find which papers have downloadable full text, download one as GROBID XML, and turn it into **sections → paragraphs → sentences** so the full text can be searched for evidence later.

**Steps:** setup → check full-text availability → download (with caching) → parse the XML.

Devlog for this notebook: `03_fulltext_extraction.devlog.md`

## 1. Setup
Same key loading and claim as notebooks 01–02.

In [1]:
import os
import requests

api_key = os.environ.get("OPENALEX_API_KEY")
print("Key loaded:", api_key is not None)  # True/False only, never the key

URL = "https://api.openalex.org/works"
HEADERS = {"Authorization": f"Bearer {api_key}"}  # key in a header, not the URL

claim = "Auditory attention decoding can be decoded using EEG"

Key loaded: True


## 2. Which papers have full text?
Semantic search, then check `has_content` for each paper.
- `XML` = GROBID XML available (PDF already split into sections/paragraphs/sentences)
- `PDF` = PDF available
- Semantic search does **not** support the `has_content` filter (→ `400` error), so we check per paper afterwards. This also avoids silently dropping relevant papers without full text.
- Result for this claim: only **7 of 20** papers had full text.

In [ ]:
params = {
    "search.semantic": claim,
    "per_page": 20,                     # ask for more, since some won't have full text
    "select": "id,display_name,publication_year,has_content,content_urls",
}
r = requests.get(URL, params=params, headers=HEADERS, timeout=30)
print("Status:", r.status_code)
if r.status_code != 200:
    print(r.json())
else:
    d = r.json()
    for w in d["results"]:
        hc = w.get("has_content") or {}
        xml = "XML" if hc.get("grobid_xml") else "---"      # full text available?
        pdf = "PDF" if hc.get("pdf") else "---"             # full text available?
        print(xml, pdf, "|", w["publication_year"], "|", w["display_name"][:75])

Status: 200
XML PDF | 2016 | Auditory-Inspired Speech Envelope Extraction Methods for Improved EEG-Based
XML PDF | 2022 | STAnet: A Spatiotemporal Attention Network for Decoding Auditory Spatial At
XML PDF | 2018 | EEG-based auditory attention detection: boundary conditions for background 
--- --- | 2022 | Detecting the locus of auditory attention based on the spectro-spatial-temp
--- --- | 2016 | Auditory attention decoding with EEG recordings using noisy acoustic refere
--- --- | 2019 | Impact of Different Acoustic Components on EEG-Based Auditory Attention Dec
XML PDF | 2020 | Decoding the Attended Speaker From EEG Using Adaptive Evaluation Intervals 
--- --- | 2019 | EEG-Based Decoding of Auditory Attention to a Target Instrument in Polyphon
--- --- | 2021 | Auditory attention decoding from electroencephalography based on long short
XML PDF | 2021 | EEG-based auditory attention decoding using speech-level-based segmented co
XML PDF | 2023 | Cortical Auditory Attention Decoding Duri

In [7]:
d["results"][0]["content_urls"]["pdf"]  # download links live in content_urls (pdf / grobid_xml)

'https://content.openalex.org/works/W2408166852.pdf'

## 3. Download the XML (once)
- `content_urls["grobid_xml"]` = download link, **$0.01 per file**
- `os.makedirs` creates `data/fulltext/` if missing
- **Caching:** if the file already exists, reuse it instead of paying again
- `data/` must be in `.gitignore`: papers keep their copyright

In [8]:
import os

paper = d["results"][0]                          # use the SAME paper for URL and filename
#print(paper)
work_id = paper["id"].split("/")[-1]             # "https://openalex.org/W44..." -> "W44..."
#print(work_id)
xml_url = paper["content_urls"]["grobid_xml"]
#print(xml_url)

folder = "../data/fulltext"
os.makedirs(folder, exist_ok=True)               # create the folder if it doesn't exist
path = f"{folder}/{work_id}.grobid.xml"

if os.path.exists(path):                         # already downloaded -> don't pay again
    print("Using saved file:", path)
else:
    x = requests.get(xml_url, headers=HEADERS, timeout=60)   # costs $0.01
    print("Status:", x.status_code)
    if x.status_code == 200:
        with open(path, "wb") as f:
            f.write(x.content)
        print("Saved:", path)
    else:
        print(x.text[:500])                      # see why it failed

Using saved file: ../data/fulltext/W2408166852.grobid.xml


## 4. Parse the XML
GROBID XML (TEI format) structure:
```
<Header>  title, authors, keywords, <abstract>
<body>       <div> = section → <head> heading, <p> paragraph → <s> sentence
             <figure> captions (at the end of the body)
<back>       acknowledgements, appendix, references
```
Parsing needs `beautifulsoup4` + `lxml`.

### First attempt (superseded by section 5)
`find_all("div")` searches **every** level, so it also picks up the `<div>`s inside figure captions. That made figure text appear after the Conclusion.

In [9]:
from bs4 import BeautifulSoup

with open(path, "rb") as f:
    soup = BeautifulSoup(f, "xml")               # "xml" parser, needs lxml

title = soup.find("titleStmt").find("title").get_text(strip=True)
abstract_tag = soup.find("abstract")
abstract = abstract_tag.get_text(" ", strip=True) if abstract_tag else None

sections = []
for div in soup.find("body").find_all("div"):    # each <div> = one section
    head = div.find("head")
    paras = [p.get_text(" ", strip=True) for p in div.find_all("p")]
    if paras:
        sections.append({"heading": head.get_text(strip=True) if head else "(no heading)",
                         "paragraphs": paras})

print("Title:", title)
print("Abstract:", (abstract or "(none)")[:300], "\n")
print("Sections:", len(sections))
for s in sections:
    print(f'  {s["heading"][:50]:50} | {len(s["paragraphs"])} paragraphs')

Title: Auditory-inspired speech envelope extraction methods for improved EEG-based auditory attention detection in a cocktail party scenario
Abstract: for improved EEG-based auditory attention detection in a cocktail party scenario 

Sections: 26
  I. INTRODUCTION                                    | 12 paragraphs
  II. AUDITORY ATTENTION DETECTION                   | 1 paragraphs
  A. Problem statement                               | 1 paragraphs
  sa (t)).                                           | 3 paragraphs
  B. Design of the decoder                           | 7 paragraphs
  C. Training                                        | 12 paragraphs
  D. Detection details                               | 1 paragraphs
  III. ENVELOPE EXTRACTION METHODS                   | 1 paragraphs
  A. Basic envelope extraction                       | 3 paragraphs
  B. Compressed envelopes                            | 2 paragraphs
  C. Subband envelopes                               | 3 paragraphs
  D

### The abstract
For this paper GROBID's `<abstract>` holds only part of the title: the real abstract is **missing** from the XML. 

In [12]:
abstract_tag = soup.find("abstract")         # GROBID puts the abstract in <abstract>
abstract = abstract_tag.get_text(" ", strip=True) if abstract_tag else None
print(abstract)

for improved EEG-based auditory attention detection in a cocktail party scenario


### Title, authors, keywords, sentences
These parts were extracted correctly. `<s>` tags = GROBID has already split the text into sentences.

In [22]:
title = soup.find("titleStmt").find("title").get_text(strip=True)
authors = [p.get_text(" ", strip=True) for p in soup.find("sourceDesc").find_all("persName")]
keywords = [t.get_text(strip=True) for t in soup.find("keywords").find_all("term")]
sentences = [s.get_text(" ", strip=True) for s in soup.find("body").find_all("s")]  # ALL sentences, incl. figure captions

print("Title:", title)
print("Authors:", authors)
print("Keywords:", keywords)
print("Body sentences:", len(sentences))
print(sentences[326])  # the last one: a side note, not the Conclusion

Title: Auditory-inspired speech envelope extraction methods for improved EEG-based auditory attention detection in a cocktail party scenario
Authors: ['Wouter Biesmans', 'Neetha Das', 'Tom Francart', 'Alexander Bertrand']
Keywords: ['Wouter Biesmans (2016)', 'Auditory-inspired speech envelope extraction methods Neuro-steered auditory prostheses', 'cocktail party', 'auditory attention', 'EEG processing', 'speech envelope', 'auditory models']
Body sentences: 327
It should be noted that [1] was based on a different set of measurements, independent of the measurements used in this manuscript


## 5. Final extraction: body → sections → paragraphs → sentences
Only take the `<div>`s **directly** inside `<body>` (`recursive=False`), so figures are skipped. Result:
```
sections[i]["heading"]           → "VI. RESULTS"
sections[i]["paragraphs"][j]     → paragraph j (a list of sentences)
sections[i]["paragraphs"][j][k]  → sentence k
```
Every piece of evidence can then be located exactly: section, paragraph, sentence.

In [23]:
from bs4 import BeautifulSoup

path = "../data/fulltext/W2408166852.grobid.xml"
with open(path, "rb") as f:
    soup = BeautifulSoup(f, "xml")

# Step 1: the body, the main text of the paper (no title/abstract, no references)
body = soup.find("body")

# Step 2: each section = a <div> directly inside <body> (figures are skipped)
sections = []
for div in body.find_all("div", recursive=False):
    head = div.find("head")
    heading = head.get_text(" ", strip=True) if head else "(no heading)"

    # Step 3: paragraphs (<p>) in this section, and sentences (<s>) in each paragraph
    paragraphs = []
    for p in div.find_all("p"):
        sentences = [s.get_text(" ", strip=True) for s in p.find_all("s")]
        paragraphs.append(sentences)

    sections.append({"heading": heading, "paragraphs": paragraphs})

# Overview
print("Sections:", len(sections))
for i, s in enumerate(sections):
    n_sent = sum(len(p) for p in s["paragraphs"])
    print(f'{i:2} | {s["heading"][:45]:45} | {len(s["paragraphs"])} paragraphs | {n_sent} sentences')

Sections: 23
 0 | I. INTRODUCTION                               | 12 paragraphs | 54 sentences
 1 | II. AUDITORY ATTENTION DETECTION              | 1 paragraphs | 1 sentences
 2 | A. Problem statement                          | 1 paragraphs | 6 sentences
 3 | sa (t)).                                      | 3 paragraphs | 8 sentences
 4 | B. Design of the decoder                      | 7 paragraphs | 13 sentences
 5 | C. Training                                   | 12 paragraphs | 35 sentences
 6 | D. Detection details                          | 1 paragraphs | 6 sentences
 7 | III. ENVELOPE EXTRACTION METHODS              | 1 paragraphs | 7 sentences
 8 | A. Basic envelope extraction                  | 3 paragraphs | 11 sentences
 9 | B. Compressed envelopes                       | 2 paragraphs | 10 sentences
10 | C. Subband envelopes                          | 3 paragraphs | 16 sentences
11 | D. Auditory models                            | 5 paragraphs | 19 sentences
12 | E. Filtering 

### Look inside one section

In [29]:
sec = sections[0]                        # pick a section by its number from the overview
print(sec["heading"], "\n")
for sent in sec["paragraphs"][:]:         # each item = one paragraph (a list of sentences)
    print("-", sent)

I. INTRODUCTION 

- ['The human auditory system has the remarkable ability to attend to one speaker and ignore the others in a so-called cocktail party scenario with multiple simultaneous speakers.', 'Since the effect was first described in 1953 [2] , it has been a topic of ongoing research in the fields of neuroscience and audiology.', "It was demonstrated in [3] that speech spectrograms that were reconstructed from cortical responses to a multispeaker stimulus reveal spectral and temporal features of the attended speaker, as if the unattended speakers weren't there.", 'This is very interesting from a neuroscientific point of view as it provides a new research tool that can help to understand and map the human auditory processing.', 'Furthermore, it allows to detect to which speaker a subject is attending in a cocktail party scenario [4] - [6] .', 'This auditory attention detection (AAD) paradigm might lead to a breakthrough for auditory prostheses (APs) such as hearing aids and cochl